In [1]:
!pip install "datasets<3.0.0" soundfile transformers accelerate torch resemblyzer librosa pandas scipy

In [9]:
import os
import numpy as np
import torch
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav

def run_evaluation(dataset_name, samples_list, text_key):
    """
    Evaluates TTS performance by comparing generated audio to reference audio.
    - dataset_name: Label for the dataset
    - samples_list: List of audio samples
    - text_key: The dictionary key for the transcript text
    """
    base_dir = f"mushra_{dataset_name}"
    # Removed 'anchor' from directory creation
    for sub in ["reference", "generated"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    for i in range(len(samples_list)):
        try:
            sample = samples_list[i]
            text = sample[text_key]
            ref_array = np.array(sample['audio']['array']).astype(np.float32)
            ref_sr = sample['audio']['sampling_rate']

            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"

            sf.write(ref_path, ref_array, ref_sr)

            # TTS Generation
            inputs = tokenizer(text, return_tensors="pt")
            with torch.no_grad():
                output = model(**inputs)
            gen_speech = output.waveform[0].cpu().numpy()
            gen_sr = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # Anchor generation removed per request

            # Embedding Similarity (Resemblyzer)
            wav_ref = preprocess_wav(Path(ref_path))
            wav_gen = preprocess_wav(Path(gen_path))

            mid = len(wav_ref) // 2
            emb_ref_full = encoder.embed_utterance(wav_ref)
            emb_ref_a = encoder.embed_utterance(wav_ref[:mid])
            emb_ref_b = encoder.embed_utterance(wav_ref[mid:])
            emb_gen = encoder.embed_utterance(wav_gen)

            ref_gen_sim = np.dot(emb_ref_full, emb_gen) / (np.linalg.norm(emb_ref_full) * np.linalg.norm(emb_gen))
            ref_self_sim = np.dot(emb_ref_a, emb_ref_b) / (np.linalg.norm(emb_ref_a) * np.linalg.norm(emb_ref_b))

            results.append({
                "sample_id": i,
                "urdu_text": text,
                "audio_duration_sec": round(len(ref_array)/ref_sr, 2),
                "ref_gen_similarity": float(ref_gen_sim),
                "ref_self_similarity": float(ref_self_sim)
            })
            print(f"[{dataset_name.upper()}] Sample {i+1}: sim={ref_gen_sim:.4f} | self={ref_self_sim:.4f}")

        except Exception as e:
            print(f"Error in {dataset_name} sample {i}: {e}")
            continue

    return results

In [10]:
from transformers import VitsModel, AutoTokenizer
import torch

MODEL_NAME = "facebook/mms-tts-urd-script_arabic"

print(f"Loading model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = VitsModel.from_pretrained(MODEL_NAME)
model.eval()

# Initialize Resemblyzer encoder
from resemblyzer import VoiceEncoder
encoder = VoiceEncoder()

print("Models loaded successfully.")

Loading model: facebook/mms-tts-urd-script_arabic...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Loaded the voice encoder model on cuda in 0.48 seconds.
Models loaded successfully.


In [11]:
NUM_SAMPLES = 30
from datasets import load_dataset
# Use streaming=True to fetch samples instantly
ds_fleurs_stream = load_dataset("google/fleurs", "ur_pk", split="test", trust_remote_code=True, streaming=True)

# Take first 15 samples from the stream
fleurs_samples = []
for sample in ds_fleurs_stream:
    fleurs_samples.append(sample)
    if len(fleurs_samples) >= NUM_SAMPLES:
        break

results_fleurs = run_evaluation("fleurs", fleurs_samples, "transcription")

Filtering for male speakers in FLEURS train set...
Found 30 male samples. Running evaluation...
[FLEURS_MALE] Sample 1: sim=0.6744 | self=0.8745
[FLEURS_MALE] Sample 2: sim=0.6853 | self=0.9393
[FLEURS_MALE] Sample 3: sim=0.7100 | self=0.9388
[FLEURS_MALE] Sample 4: sim=0.7086 | self=0.8488
[FLEURS_MALE] Sample 5: sim=0.7159 | self=0.9477
[FLEURS_MALE] Sample 6: sim=0.6139 | self=0.8957
[FLEURS_MALE] Sample 7: sim=0.7455 | self=0.8697
[FLEURS_MALE] Sample 8: sim=0.6695 | self=0.9042
[FLEURS_MALE] Sample 9: sim=0.6725 | self=0.8804
[FLEURS_MALE] Sample 10: sim=0.6646 | self=0.9442
[FLEURS_MALE] Sample 11: sim=0.6846 | self=0.9601
[FLEURS_MALE] Sample 12: sim=0.6451 | self=0.8843
[FLEURS_MALE] Sample 13: sim=0.7068 | self=0.8453
[FLEURS_MALE] Sample 14: sim=0.6686 | self=0.9201
[FLEURS_MALE] Sample 15: sim=0.7202 | self=0.8518
[FLEURS_MALE] Sample 16: sim=0.7114 | self=0.8746
[FLEURS_MALE] Sample 17: sim=0.6234 | self=0.8306
[FLEURS_MALE] Sample 18: sim=0.5776 | self=0.9511
[FLEURS_MALE]

In [12]:
import pandas as pd
import zipfile
import os

# 1. Prepare the data for CSV
all_results = []
# Assuming results_fleurs exists from previous execution
try:
    for entry in results_fleurs:
        all_results.append({
            "dataset": "fleurs",
            "sample_id": entry["sample_id"],
            "urdu_text": entry["urdu_text"],
            "audio_duration_sec": entry["audio_duration_sec"],
            "ref_gen_similarity": entry["ref_gen_similarity"]
        })
except NameError:
    print("Warning: results_fleurs not found. Please ensure the evaluation cell has finished running.")

# 2. Save to CSV
csv_filename = "evaluation_results.csv"
df = pd.DataFrame(all_results)
df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
print(f"Created {csv_filename}")

# 3. Create Zip file
zip_filename = "mushra_results_package.zip"
datasets_to_zip = ["fleurs", "fleurs_male"] # Add others if they were generated

with zipfile.ZipFile(zip_filename, "w") as zf:
    # Add the CSV
    zf.write(csv_filename)

    # Add the audio folders
    for d_name in datasets_to_zip:
        folder = f"mushra_{d_name}"
        if os.path.exists(folder):
            for root, dirs, files in os.walk(folder):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Preserve folder structure inside zip
                    zf.write(file_path, os.path.relpath(file_path, "."))

print(f"Successfully created {zip_filename} containing results and audio samples.")

Created evaluation_results.csv
Successfully created mushra_results_package.zip containing results and audio samples.
